# Phase 2/3 Training Pipeline

Train the network

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import os, sys
import pandas as pd
from pathlib import Path
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1" # evita il warning

sys.path.append(str(Path("..").resolve()))


from src.data import make_loaders
from src.config import MODEL_CONFIGS
from src.model import BoneAgeCNN


d:\magistrale unipi\Computing methods\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2.1 Data loaders (recap)

In [4]:
loaders, scaler = make_loaders() # bach size fixed = 32

for name, ld in loaders.items():
    print(f'{name:5s}: {len(ld.dataset):6d} images, {len(ld):4d} batches')
img, gen, tgt = next(iter(loaders['train']))
print('batch shapes -> image', tuple(img.shape), '| gender', tuple(gen.shape), '| target', tuple(tgt.shape))

train:  12611 images,  394 batches
valid:    800 images,   25 batches
test :    625 images,   20 batches
batch shapes -> image (32, 1, 256, 256) | gender (32, 1) | target (32, 1)


Bone age (months) is z-scored using the training mean/std before computing the loss. This keeps the loss scale stable across configs; predictions are converted back to months for reporting via `scaler.inverse`.

Example:

In [5]:
print(f'target mean={scaler.mean:.1f} months, std={scaler.std:.1f} months')
print('example 120 months ->', round(scaler.transform(120.0), 3), '-> back to',
      round(scaler.inverse(scaler.transform(120.0)), 1))

target mean=127.3 months, std=41.2 months
example 120 months -> -0.178 -> back to 120.0


## 2.2  Model selection (we try a few different combinations)

We compare **4 configurations** to show the role of tuning.

In [6]:
pd.DataFrame(MODEL_CONFIGS)

,baseline,high_dropout,shallow,adam
in_channels,1,1,1,1
fc_hidden,512,512,512,512
img_size,256,256,256,256
batch_size,32,32,32,32
epochs,30,30,30,30
weight_decay,0.00005,0.00005,0.00005,0.00005
momentum,0.9,0.9,0.9,0.9
lr_decay,0.97,0.97,0.97,0.97
n_blocks,5,5,4,5
base_filters,8,8,8,8


## 2.3 Training


In [ ]:
for name, base in MODEL_CONFIGS.items():
    cfg = dict(base)
    cfg['name'] = name

    print("training", name)
    model = BoneAgeCNN(
        n_blocks=cfg["n_blocks"],
        base_filters=cfg["base_filters"],
        in_channels=cfg["in_channels"],

        fc_hidden=cfg["fc_hidden"],
        dropout=cfg["dropout"],
        img_size=cfg["img_size"],
    )
    for name, base in MODEL_CONFIGS.items():
        cfg = dict(base)
        cfg['name'] = name

        print("training", name)
        model = BoneAgeCNN(
            n_blocks=cfg["n_blocks"],
            base_filters=cfg["base_filters"],
            in_channels=cfg["in_channels"],

            fc_hidden=cfg["fc_hidden"],
            dropout=cfg["dropout"],
            img_size=cfg["img_size"],
        )

        device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
        model.fit(loaders, cfg, device, name)
    

training baseline


RuntimeError: PyTorch is not linked with support for mps devices